In [1]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling, TrainingArguments, Trainer

d:\miniconda3\envs\PyTorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds = Dataset.load_from_disk("./wiki_cn_filtered/")
ds

Dataset({
    features: ['source', 'completion'],
    num_rows: 10000
})

In [4]:
ds[0]

{'source': 'wikipedia.zh2307',
 'completion': "西安交通大学博物馆（Xi'an Jiaotong University Museum）是一座位于西安交通大学的博物馆，馆长是锺明善。\n历史\n2004年9月20日开始筹建，2013年4月8日正式建成开馆，位于西安交通大学兴庆校区陕西省西安市咸宁西路28号。建筑面积6,800平米，展厅面积4,500平米，馆藏文物4,900余件。包括历代艺术文物馆、碑石书法馆、西部农民画馆、邢良坤陶瓷艺术馆、陕西秦腔博物馆和书画展厅共五馆一厅。\n营业时间\n* 周一至周六：上午九点至十二点，下午一点至五点\n* 周日闭馆"}

In [5]:
tokenizer = AutoTokenizer.from_pretrained("hfl/chinese-macbert-base")

def process_func(examples):
    return tokenizer(examples["completion"], max_length=384, truncation=True)

In [6]:
tokenized_ds = ds.map(process_func, batched=True, remove_columns=ds.column_names)
tokenized_ds

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 10000
})

In [7]:
print(tokenized_ds[0])

{'input_ids': [101, 6205, 2128, 769, 6858, 1920, 2110, 1300, 4289, 7667, 8020, 13135, 112, 9064, 12095, 8731, 8626, 8181, 8736, 10553, 8021, 3221, 671, 2429, 855, 754, 6205, 2128, 769, 6858, 1920, 2110, 4638, 1300, 4289, 7667, 8024, 7667, 7270, 3221, 7247, 3209, 1587, 511, 1325, 1380, 8258, 2399, 130, 3299, 8113, 3189, 2458, 1993, 5040, 2456, 8024, 8138, 2399, 125, 3299, 129, 3189, 3633, 2466, 2456, 2768, 2458, 7667, 8024, 855, 754, 6205, 2128, 769, 6858, 1920, 2110, 1069, 2412, 3413, 1277, 7362, 6205, 4689, 6205, 2128, 2356, 1496, 2123, 6205, 6662, 8143, 1384, 511, 2456, 5029, 7481, 4916, 127, 117, 8280, 2398, 5101, 8024, 2245, 1324, 7481, 4916, 125, 117, 8195, 2398, 5101, 8024, 7667, 5966, 3152, 4289, 125, 117, 8567, 865, 816, 511, 1259, 2886, 1325, 807, 5686, 3318, 3152, 4289, 7667, 510, 4811, 4767, 741, 3791, 7667, 510, 6205, 6956, 1093, 3696, 4514, 7667, 510, 6928, 5679, 1787, 7378, 4487, 5686, 3318, 7667, 510, 7362, 6205, 4912, 5579, 1300, 4289, 7667, 1469, 741, 4514, 2245, 1324,

In [8]:
from torch.utils.data import DataLoader

dl = DataLoader(tokenized_ds, batch_size=16, collate_fn=DataCollatorForLanguageModeling(tokenizer, mlm=True, mlm_probability=0.15))

In [9]:
next(enumerate(dl))

(0,
 {'input_ids': tensor([[  101,  6205,  2128,  ...,     0,     0,     0],
         [  101,  1762,  1921,  ...,  6381,   103,   102],
         [  101,  3330,  1984,  ..., 10596,   119,   102],
         ...,
         [  101,  2861,  5838,  ...,     0,     0,     0],
         [  101,  1266,   776,  ...,  1061,   103,   102],
         [  101,   677,  5862,  ...,  7188,  1146,   102]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]]), 'labels': tensor([[-100, -100, -100,  ..., -100, -100, -100],
         [-100, 1762, -100,  ..., -100, 6770, -100],
         [-

In [10]:
tokenizer.mask_token, tokenizer.mask_token_id

('[MASK]', 103)

In [11]:
model = AutoModelForMaskedLM.from_pretrained("hfl/chinese-macbert-base")

Some weights of the model checkpoint at hfl/chinese-macbert-base were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [12]:
args = TrainingArguments(
    output_dir="./masked_lm",
    per_device_train_batch_size=16,
    logging_steps=200,
    num_train_epochs=2
)

In [13]:
trainer = Trainer(
    args=args,
    model=model,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=True, mlm_probability=0.15)
)

C:\Users\10433\AppData\Local\Temp\ipykernel_12180\3946062033.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [14]:
trainer.train()

Step,Training Loss
200,1.392500
400,1.343700
600,1.334800
800,1.288000
1000,1.264300
1200,1.256700


TrainOutput(global_step=1250, training_loss=1.3105886840820313, metrics={'train_runtime': 612.0247, 'train_samples_per_second': 32.678, 'train_steps_per_second': 2.042, 'total_flos': 3947639316480000.0, 'train_loss': 1.3105886840820313, 'epoch': 2.0})

In [15]:
from transformers import pipeline

pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer, device=0)

Device set to use cuda:0


In [16]:
pipe("西安交通[MASK][MASK]博物馆（Xi'an Jiaotong University Museum）是一座位于西安交通大学的博物馆")

[[{'score': 0.9982884526252747,
   'token': 1920,
   'token_str': '大',
   'sequence': "[CLS] 西 安 交 通 大 [MASK] 博 物 馆 （ xi ' an jiaotong university museum ） 是 一 座 位 于 西 安 交 通 大 学 的 博 物 馆 [SEP]"},
  {'score': 0.001050662132911384,
   'token': 2110,
   'token_str': '学',
   'sequence': "[CLS] 西 安 交 通 学 [MASK] 博 物 馆 （ xi ' an jiaotong university museum ） 是 一 座 位 于 西 安 交 通 大 学 的 博 物 馆 [SEP]"},
  {'score': 3.801402999670245e-05,
   'token': 5466,
   'token_str': '职',
   'sequence': "[CLS] 西 安 交 通 职 [MASK] 博 物 馆 （ xi ' an jiaotong university museum ） 是 一 座 位 于 西 安 交 通 大 学 的 博 物 馆 [SEP]"},
  {'score': 2.845884046109859e-05,
   'token': 2339,
   'token_str': '工',
   'sequence': "[CLS] 西 安 交 通 工 [MASK] 博 物 馆 （ xi ' an jiaotong university museum ） 是 一 座 位 于 西 安 交 通 大 学 的 博 物 馆 [SEP]"},
  {'score': 2.3257707653101534e-05,
   'token': 7770,
   'token_str': '高',
   'sequence': "[CLS] 西 安 交 通 高 [MASK] 博 物 馆 （ xi ' an jiaotong university museum ） 是 一 座 位 于 西 安 交 通 大 学 的 博 物 馆 [SEP]"}],
 [{'score': 0.996

In [17]:
pipe("下面是一则[MASK][MASK]新闻。小编报道，近日，游戏产业发展的非常好！")

[[{'score': 0.09096357971429825,
   'token': 7028,
   'token_str': '重',
   'sequence': '[CLS] 下 面 是 一 则 重 [MASK] 新 闻 。 小 编 报 道 ， 近 日 ， 游 戏 产 业 发 展 的 非 常 好 ！ [SEP]'},
  {'score': 0.0710235983133316,
   'token': 3173,
   'token_str': '新',
   'sequence': '[CLS] 下 面 是 一 则 新 [MASK] 新 闻 。 小 编 报 道 ， 近 日 ， 游 戏 产 业 发 展 的 非 常 好 ！ [SEP]'},
  {'score': 0.05691039189696312,
   'token': 2031,
   'token_str': '娱',
   'sequence': '[CLS] 下 面 是 一 则 娱 [MASK] 新 闻 。 小 编 报 道 ， 近 日 ， 游 戏 产 业 发 展 的 非 常 好 ！ [SEP]'},
  {'score': 0.05566421151161194,
   'token': 5381,
   'token_str': '网',
   'sequence': '[CLS] 下 面 是 一 则 网 [MASK] 新 闻 。 小 编 报 道 ， 近 日 ， 游 戏 产 业 发 展 的 非 常 好 ！ [SEP]'},
  {'score': 0.04787662252783775,
   'token': 4178,
   'token_str': '热',
   'sequence': '[CLS] 下 面 是 一 则 热 [MASK] 新 闻 。 小 编 报 道 ， 近 日 ， 游 戏 产 业 发 展 的 非 常 好 ！ [SEP]'}],
 [{'score': 0.07257771492004395,
   'token': 7319,
   'token_str': '闻',
   'sequence': '[CLS] 下 面 是 一 则 [MASK] 闻 新 闻 。 小 编 报 道 ， 近 日 ， 游 戏 产 业 发 展 的 非 常 好 ！ [SEP]'},
  {'